# Import Libraries and implement logger

In [41]:
## IMPORT LIBRARIES
import logging
import numpy as np
import pandas as pd
import torch

from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from io import StringIO
from datetime import datetime
from typing import Tuple, List, Optional, Dict
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

In [33]:
# CREATE LOGGER FOR JUPYTER NOTEBOOKS
def setup_logger():
    # Create logger instance
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)

    # Clear existing, avoid duplicate logs
    if logger.hasHandlers():
        logger.handlers.clear()
    
    # Create FileHandler in overrite mode
    file_handler = logging.FileHandler('../../logmodeler.log', mode ='w')

    # Set format
    formatter = logging.Formatter('%(asctime)s - %(levelno)s - %(lineno)d - %(module)s - %(message)s')
    file_handler.setFormatter(formatter)

    # Add FileHandler to logger
    logger.addHandler(file_handler)

    return logger

logger = setup_logger()

# Inspcet the data read in after the Visualzer

In [34]:
## Define function for inspection
def inspect_raw_data(df, output_path=None):
    '''
    Inspect a DataFrame and save info to a text file.
    Parameters:
    df : pandas DataFrame
    output_path : Path or str, optional
        Where to save the output file. If None and save_to_file is True,
        will use current directory
    '''

    # Set display options
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    
    # Prepare the information
    info_text = []
    info_text.append(f"Generated on: {datetime.now()}")
    info_text.append(f"Shape: {df.shape}")
    
    # Get info in string format
    buffer = StringIO()
    df.info(buf=buffer)
    info_text.append(f"\nDataFrame Info:")
    info_text.append(buffer.getvalue())

    # Add first 5 rows
    buffer_head = StringIO()
    df.head().to_string(buf=buffer_head)
    info_text.append(f"\nFirst 5 rows:")
    info_text.append(buffer_head.getvalue())

    # Add last 5 rows
    buffer_tail = StringIO()
    df.tail().to_string(buf=buffer_tail)
    info_text.append(f"\nLast 5 rows:")
    info_text.append(buffer_tail.getvalue())
    
    # Join all information
    full_report = '\n'.join(info_text)
    output_path = Path(output_path)
    output_path.write_text(full_report)
    logger.info(f"Report saved to: {output_path}")
    
    return full_report

In [35]:
# READ IN
data_path_eso = Path('..') / '..' / 'data' / 'processed' / 'Visualizer' / 'eso'
data_path_ger = Path('..') / '..' / 'data' / 'processed' / 'Visualizer' / 'ger'

# ESO
# Normalized
X_train_eso_scaled = pd.read_csv(data_path_eso / 'X_train_eso_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
X_test_eso_scaled = pd.read_csv(data_path_eso / 'X_test_eso_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
# Not normalized
y_train_eso = pd.read_csv(data_path_eso / 'y_train_eso.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
y_test_eso = pd.read_csv(data_path_eso / 'y_test_eso.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']

# GER
# Normalized
X_train_ger_scaled = pd.read_csv(data_path_ger / 'X_train_ger_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
X_test_ger_scaled = pd.read_csv(data_path_ger / 'X_test_ger_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
# Not normalized
y_train_ger = pd.read_csv(data_path_ger / 'y_train_ger.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
y_test_ger = pd.read_csv(data_path_ger / 'y_test_ger.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']

In [36]:
"""## SAVE FORE INSPECTION
# ESO
# Normalized
eso_scaled_Xtrain = 'X_train_eso_scaled'; eso_scaled_Xtest = 'X_test_eso_scaled'
_ = inspect_raw_data(X_train_eso_scaled, output_path=Path(f'{data_path}/../Modeler/eso/{eso_scaled_Xtrain}.txt'))
_ = inspect_raw_data(X_test_eso_scaled, output_path=Path(f'{data_path}/../Modeler/eso/{eso_scaled_Xtest}.txt'))
# Not normalized
eso_ytrain = 'y_train_eso'; eso_ytest = 'y_test_eso'
_ = inspect_raw_data(y_train_eso, output_path=Path(f'{data_path}/../Modeler/eso/{eso_ytrain}.txt'))
_ = inspect_raw_data(y_test_eso, output_path=Path(f'{data_path}/../Modeler/eso/{eso_ytest}.txt'))
# GER
# Normalized
ger_scaled_Xtrain = 'X_train_ger_scaled'; ger_scaled_Xtest = 'X_test_ger_scaled'
_ = inspect_raw_data(X_train_ger_scaled, output_path=Path(f'{data_path}/../Modeler/ger/{ger_scaled_Xtrain}.txt'))
_ = inspect_raw_data(X_test_ger_scaled, output_path=Path(f'{data_path}/../Modeler/ger/{ger_scaled_Xtest}.txt'))
# Not normalized
ger_ytrain = 'y_train_ger'; ger_ytest = 'y_test_ger'
_ = inspect_raw_data(y_train_ger, output_path=Path(f'{data_path}/../Modeler/ger/{ger_ytrain}.txt'))
_ = inspect_raw_data(y_test_ger, output_path=Path(f'{data_path}/../Modeler/ger/{ger_ytest}.txt'))"""

"## SAVE FORE INSPECTION\n# ESO\n# Normalized\neso_scaled_Xtrain = 'X_train_eso_scaled'; eso_scaled_Xtest = 'X_test_eso_scaled'\n_ = inspect_raw_data(X_train_eso_scaled, output_path=Path(f'{data_path}/../Modeler/eso/{eso_scaled_Xtrain}.txt'))\n_ = inspect_raw_data(X_test_eso_scaled, output_path=Path(f'{data_path}/../Modeler/eso/{eso_scaled_Xtest}.txt'))\n# Not normalized\neso_ytrain = 'y_train_eso'; eso_ytest = 'y_test_eso'\n_ = inspect_raw_data(y_train_eso, output_path=Path(f'{data_path}/../Modeler/eso/{eso_ytrain}.txt'))\n_ = inspect_raw_data(y_test_eso, output_path=Path(f'{data_path}/../Modeler/eso/{eso_ytest}.txt'))\n# GER\n# Normalized\nger_scaled_Xtrain = 'X_train_ger_scaled'; ger_scaled_Xtest = 'X_test_ger_scaled'\n_ = inspect_raw_data(X_train_ger_scaled, output_path=Path(f'{data_path}/../Modeler/ger/{ger_scaled_Xtrain}.txt'))\n_ = inspect_raw_data(X_test_ger_scaled, output_path=Path(f'{data_path}/../Modeler/ger/{ger_scaled_Xtest}.txt'))\n# Not normalized\nger_ytrain = 'y_train_

# Begin modeling after read in the dataframes 
#### Set requirements for building a predictor 
    - Use pytorch LSTM
      Consider: https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html, https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
    - Use dataclasses, where the model is build, name it Modeler 
    - Consider the figure of the modeling process
    - Use the already split and normalized dataframes X_train_XXX_scaled, X_test_XXX_scaled, y_train_XXX, y_test_XXX
    - Consider the information in the text files and that there is also are 8 files all over, XXX is replaced with eso or ger
    - Use cross Validation
      Consider: https://scikit-learn.org/1.5/modules/cross_validation.html
    - Use proper Evaluation Metric
    - Consider: batch size, epoche, learning rate, learning rate dk

In [39]:
## SET FIXED VARIABLES FOR ESO AND GER
class DatasetType(Enum):
    ESO = 'eso'
    GER = 'ger'

In [50]:
## CONFIGURATIONS FOR THE MODEL
@dataclass
class ModelConfig:
    # INITIALIZATION 
    dataset_type: DatasetType # Declaration of used type of dataset
    input_size: int = field(init=False) # Number of expected features in the input x
    hidden_size: int = 128 # Number of features in the hidden state h
    num_layers: int = 1 # Number of recurrent layers (with 2, 2th would output from 1st)
    
    #bias: True # Use bias weights b_ih and b_hh (default:True)
    #batch_first: True # I/O tensors provided as (batch, seq, feature for True) and (seq, batch, feature for False) 
    dropout: float = 0.2 # introduces a dropout layer on the output (expect last layer) (default=0)
    #bidirectional: False # (default=False)
    #proj_size: int = 0 #  if>0, will use LSTM with projections of corresponding size. (default=0)

    output_size: int = 1 # Size of output
    batch_size: int = 32 # Number of samples processed together in one forward/backward pass during training.
    epochs: int = 10 # Number of epochs
    learning_rate: float = 0.001 # step size directed gradient
    lr_decay: float = 0.95 
    sequence_length: int = 24 # Number of steps to look back
    n_splots: int = 2 # Number of folds for cross validation

    def __post_init__(self):
        self.input_size = 86 if self.dataset_type == DatasetType.ESO else 67


In [51]:
## DEFINE SEQUENCE
class EnergyDataset(Dataset):
    # Transform to numpy for numerical or tensor-based operations
    def __init__(self, X:np.ndarray, y:np.ndarray, sequence_length: int):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.sequence_length = sequence_length

    # Get sequence length
    def __len__(self):
        return len(self.X) - self.sequence_length + 1 
    
    # 
    def __getitem__(self, idx):
        x_sequence = self.X[idx:idx + self.sequence_length] # Input sequence (sequence_length steps starting at idx)
        y_target = self.y[idx + self.sequence_length - 1] # Target value (last step of the sequence)
        '''
        self.X[0:0 + 24] extracts the first 24 elements (idx 0...23) of self.X
        self.y[0 + 24 - 1] extracts the 24th element (idx 23) of self.y
        '''
        return x_sequence, y_target

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__() # Call method from a parent (super) class
        self.lstm = nn.LSTM(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            batch_first=True,
            dropout=config.dropout
        )
        self.fc = nn.Linear(config.hidden_size, config.output_size) # Linear only for final prediction
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x) # Pass input through LSTM layer
        predictions = self.fc(lstm_out[:, -1, :]) # Pass the last time steps output through the fully connected layer
        return predictions        

self.fc = nn.Linear(config.hidden_size, config.output_size) \
$\rightarrow$ Defines a fully connected (fc) layer. 
Creates a linear transformation of the input features to the output features: 
$y=x~W^T+b$

* $x$ is the input tensor of size (batch_size, in_features)
* $W$ is the weight matrix of shape (out_features, in_features)
* $b$ is the bias vector of shape (out_features)
* $y$ is the output tensor of shape (batch_size, out_features)

config.hidden_size: The number of input features to the linear layer. After passing through the LSTM layer, the output will have a shape (batch_size, sequence_length, hidden_size) \
$\rightarrow$ https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html batch_first: True $\rightarrow$ (batch, seq, feature))

config.output_size: The number of output features of the linear layer. Represents the final dimensionality of the model output. (Typically 1 for regression)

---

$x$: input tensor with shape (batch_size, sequence_length)
* batch_size: Number of sequences in a batch. E.g. dataset with 10 000 samples: 10 000/batch_size
* sequence_length: Number of time steps in each sequence.
* input_size: Number of features at each time step

lstm_out: output features for each time step, with shape (batch_size, sequence_length, hidden_size) contains output feature ($h_t$: hidden state vector for all time steps $t$ in the sequence) \
_: The hidden and cell states. Tupel of ($h_n$: hidden state of the last time step for all layers, $c_n$: cell state of the last time step for all layers) not used here, so we discard them with _  

lstm_out[:, -1, :]: Selects the hidden state output for the last time step of each sequence
* :: Selects all batches.
* -1: Selects the last time step (index -1 refers to the last element in the sequence).
* :: Selects all features (of size hidden_size) for that time step.

$\rightarrow$ Get tensor with shape (batch_size, hidden_size)

---

1. The LSTM processes the sequence (x) with shape (batch_size, sequence_length, input_size). 
2. The output tensor (lstm_out) retains the batch_first shape, i.e., (batch_size, sequence_length, hidden_size).  
3. You extract the last time step's output for each sequence, resulting in (batch_size, hidden_size). 
4. The fully connected (self.fc) layer transforms this to (batch_size, output_size). 


In [55]:
@dataclass
class Modeler:
    config: ModelConfig # Get configurations
    device: torch.device = field(default_factory=lambda: torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    model: Optional[LSTMModel] = None # Declare model attribute, LSTMModel for training
    criterion: Optional[nn.Module] = None # For reference a loss function module (as CrossEntrupyLoss, MSELoss)
    optimizer: Optional[torch.optim.Optimizer] = None # For referencing an optimizer 
    scheduler: Optional[torch.optim.lr_scheduler.StepLR] = None # For referencing a learning rate
    '''
    With the device:
    field function is capable to specify default values and meta data for class attributes
    default_factory argument provides a function that returns default values. With lambda function it checks, if a Compute Unified Device Architecture (CUDA) device is available.
    CUDA is a Application Programming Interface (API).
    In this case it refers to the GPU. If GPU is not available, it selects the CPU if not.
    '''

    def __post_init__(self):
        self.model = LSTMModel(self.config).to(self.device)
        self.criterion = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.config.learning_rate)
        self.scheduler = torch.optim.lr_scheduler.ExponentialLR(self.optimizer, gamma=self.config.lr_decay)

    def prepare_data(self, X: pd.DataFrame, y: pd.DataFrame) -> Tuple[DataLoader, np.array]:
        # Remove ID column with datetime and convert to numpy array
        X_values = X.drop('ID', axis=1).values
        y_values = y.drop('ID', axis=1). values.reshape(-1,1)

        dataset = EnergyDataset(X_values, y_values, self.config.sequence_length)
        dataloader = DataLoader(dataset, batch_size=self.config.batch_size, shuffle=True)

        return dataloader, y_values